# 02 · MNIST model configuration

Notebook này đọc dataset `.npz`, chia train/validation, fit StandardScaler và tìm cấu hình tốt cho Logistic Regression cùng ANN tuned.

Kết quả được lưu ở `mnist_model_config.json`.

In [1]:
from pathlib import Path
import sys
from copy import deepcopy
import json
import warnings
import numpy as np

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'gk' / 'web' / 'backend').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np

from sklearn.metrics import accuracy_score
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import StandardScaler
from gk.web.backend.digits_config import (
    DATASET_DIR, DATASET_PATH, MODEL_CONFIG_PATH, PIXEL_COUNT, PIXEL_SIZE, SPLIT_PATH,MAX_ITER
)
from gk.web.backend.model_config import (
    AUGMENT_FACTOR, AUGMENT_ROTATE_DEGREES, AUGMENT_SCALE_RANGE, AUGMENT_SHARPEN,
    AUGMENT_SHIFT_PIXELS, AUGMENT_STROKE_VARIANTS, AUGMENT_TRAINING,
    MODEL_CONFIGS as DEFAULT_MODEL_CONFIGS,
)
from gk.web.backend.digits_models import (
    LOGISTIC_CANDIDATES, MLP_SEARCH_CONFIGS, RANDOM_SEED, SEARCH_MAX_ITER,
    build_logistic, build_mlp, choose_search_subset,
)
warnings.filterwarnings('ignore', category=ConvergenceWarning)

CONFIG_PATH = MODEL_CONFIG_PATH
print('Configured input:', f'{PIXEL_SIZE}x{PIXEL_SIZE} ({PIXEL_COUNT} features)')
data = np.load(DATASET_PATH)
X_train_images = data['X_train_pixels'].astype(float)
y_train = data['y_train'].astype(int)
X_test_images = data['X_test_pixels'].astype(float)
y_test = data['y_test'].astype(int)
X_train_pixels = X_train_images.reshape(len(X_train_images), -1)
X_test_pixels = X_test_images.reshape(len(X_test_images), -1)
print('Loaded:', DATASET_PATH)


Configured input: 16x16 (256 features)
Loaded: /Users/nguyenkz/Documents/code/hoc/CS2207.26.1.CH.02/gk/web/backend/artifacts/mnist_dataset/16x16/mnist_normalized.npz


In [2]:
split = json.loads(SPLIT_PATH.read_text(encoding='utf-8'))
train_indices = np.asarray(split['fit_indices'], dtype=int)
validation_indices = np.asarray(split['validation_indices'], dtype=int)
scaler = StandardScaler().fit(X_train_pixels[train_indices])
X_fit = scaler.transform(X_train_pixels[train_indices])
X_validation = scaler.transform(X_train_pixels[validation_indices])
X_test = scaler.transform(X_test_pixels)
y_fit = y_train[train_indices]
y_validation = y_train[validation_indices]
print(X_fit.shape, X_validation.shape, X_test.shape)

(48000, 256) (12000, 256) (10000, 256)


## Cấu hình bốn model

Sửa cell này để thử architecture, activation, learning rate, số epoch hoặc `C`.

In [3]:
MODEL_CONFIGS = deepcopy(DEFAULT_MODEL_CONFIGS)

print(MODEL_CONFIGS)

{'logistic': {'kind': 'linear', 'C': 1.0, 'solver': 'lbfgs', 'max_iter': 300}, 'mlp_4_one_layer': {'kind': 'ann', 'hidden_layer_sizes': (8,), 'activation': 'tanh', 'learning_rate_init': 0.001, 'batch_size': 256, 'max_iter': 350, 'early_stopping': False}, 'mlp_4_two_layers': {'kind': 'ann', 'hidden_layer_sizes': (8, 8), 'activation': 'tanh', 'learning_rate_init': 0.001, 'batch_size': 256, 'max_iter': 350, 'early_stopping': False}, 'compact_tuned': {'kind': 'ann', 'hidden_layer_sizes': (128, 64), 'activation': 'relu', 'learning_rate_init': 0.002, 'batch_size': 256, 'max_iter': 350, 'early_stopping': False}}


## Tuning 30 vòng

Candidate được train trên tối đa 24.000 mẫu train để thử nghiệm nhanh. Bước search có giới hạn iteration riêng; warning hội tụ được ẩn khỏi output và trạng thái `converged`/`iterations` được lưu trong JSON. Cấu hình cuối sẽ được train lại trên toàn bộ train ở notebook tiếp theo.

In [4]:
from gk.web.backend.model_config import architecture_sizes

def parameter_count(architecture):
    return sum((left * right) + right for left, right in zip(architecture[:-1], architecture[1:]))

print('Fixed configurations loaded from model_config.py:')
for model_id, model_config in MODEL_CONFIGS.items():
    architecture = architecture_sizes(model_config)
    print(f'  {model_id:24} {architecture} · parameters={parameter_count(architecture)}')
print('No candidate training or automatic tuning is performed in this notebook.')


Fixed configurations loaded from model_config.py:
  logistic                 [256, 10] · parameters=2570
  mlp_4_one_layer          [256, 8, 10] · parameters=2146
  mlp_4_two_layers         [256, 8, 8, 10] · parameters=2218
  compact_tuned            [256, 128, 64, 10] · parameters=41802
No candidate training or automatic tuning is performed in this notebook.


In [5]:
config = {
    'version': 1,
    'random_seed': RANDOM_SEED,
    'dataset_file': DATASET_PATH.name,
    'input_shape': [PIXEL_SIZE, PIXEL_SIZE],
    'feature_count': PIXEL_COUNT,
    'class_count': 10,
    'fit_indices': train_indices.astype(int).tolist(),
    'validation_indices': validation_indices.astype(int).tolist(),
    'test_indices': np.arange(len(y_test)).astype(int).tolist(),
    'scaler': {'mean': scaler.mean_.tolist(), 'std': scaler.scale_.tolist()},
    'models': MODEL_CONFIGS,
    'augmentation': {
        'enabled': AUGMENT_TRAINING, 'factor': AUGMENT_FACTOR,
        'shift_pixels': AUGMENT_SHIFT_PIXELS,
        'stroke_variants': AUGMENT_STROKE_VARIANTS,
        'scale_range': list(AUGMENT_SCALE_RANGE),
        'rotate_degrees': AUGMENT_ROTATE_DEGREES,
        'sharpen': AUGMENT_SHARPEN,
        'applied_to': 'fit split only; mnist_train.ipynb materializes variants',
    },
    'search': {
        'enabled': False,
        'sample_count': 0,
        'logistic_rounds': 0,
        'mlp_rounds': 0,
        'logistic_candidates': [],
        'mlp_candidates': [],
        'selected_logistic_validation_accuracy': None,
        'selected_mlp_validation_accuracy': None,
    },
}
CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding='utf-8')
print(f'Wrote {CONFIG_PATH}')
assert set(config['models']) == set(MODEL_CONFIGS)
assert all(config['models'][model_id].get('layers') for model_id in MODEL_CONFIGS)


Wrote /Users/nguyenkz/Documents/code/hoc/CS2207.26.1.CH.02/gk/web/backend/artifacts/mnist_dataset/16x16/mnist_model_config.json
